In [106]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
import psycopg
from dotenv import load_dotenv
import os
import mlflow
import mlflow.xgboost



load_dotenv()

True

# EDA

In [88]:
# Connect to PostgreSQL
DB_NAME = os.getenv("DB_NAME")
DB_SCHEMA = os.getenv("DB_SCHEMA")
DB_TABLE_NAME = os.getenv("DB_TABLE_NAME")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = int(os.getenv("DB_PORT", 5432))
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_USER = os.getenv("DB_USER")

with psycopg.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
) as conn:

    query = f"""
        SELECT *
        FROM {DB_NAME}.{DB_SCHEMA}.{DB_TABLE_NAME};
    """

    df = pd.read_sql(query, conn)

    conn.close()

C:\Users\lixx\AppData\Local\Temp\ipykernel_13616\325832233.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [89]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,load_date
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1,2026-10-24
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,2026-10-24
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,2026-10-24
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0,2026-10-24
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,2026-10-24


In [90]:
df.isnull().sum()

RowNumber          0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
load_date          0
dtype: int64

In [91]:
columns = [
    'CreditScore',
    'Tenure',
    'Balance',
    'NumOfProducts',
    'IsActiveMember',
    'EstimatedSalary',
    'Exited'
]

new_df = df[columns].drop_duplicates()

In [92]:
new_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CreditScore      10000 non-null  int64  
 1   Tenure           10000 non-null  int64  
 2   Balance          10000 non-null  float64
 3   NumOfProducts    10000 non-null  int64  
 4   IsActiveMember   10000 non-null  int64  
 5   EstimatedSalary  10000 non-null  float64
 6   Exited           10000 non-null  int64  
dtypes: float64(2), int64(5)
memory usage: 625.0 KB


In [93]:
df_features = new_df.drop('Exited', axis=1)
df_target = new_df['Exited']

In [94]:
#Check normalization of target variable
df_target.value_counts(normalize=True)

Exited
0    0.7963
1    0.2037
Name: proportion, dtype: float64

In [95]:
X_train, X_test, y_train, y_test = train_test_split(df_features, df_target, test_size=0.3, random_state=42)

In [96]:
print('Training set size:', X_train.shape[0])
print('Test set size:', X_test.shape[0])

Training set size: 7000
Test set size: 3000


# Create Model

In [108]:
lgbm_model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31,
    random_state=42,
    class_weight="balanced",
    verbosity=-1
)


xgb_params = {
    "n_estimators":500,
    "learning_rate":0.05,
    "max_depth":6,
    "subsample":0.8,
    "colsample_bytree":0.8,
    "objective":"binary:logistic",
    "eval_metric":"auc",
    "random_state":42,
    "n_jobs":-1
    }

xgb_model = XGBClassifier(**xgb_params)

In [98]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7000 entries, 9069 to 7270
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CreditScore      7000 non-null   int64  
 1   Tenure           7000 non-null   int64  
 2   Balance          7000 non-null   float64
 3   NumOfProducts    7000 non-null   int64  
 4   IsActiveMember   7000 non-null   int64  
 5   EstimatedSalary  7000 non-null   float64
dtypes: float64(2), int64(4)
memory usage: 382.8 KB


In [116]:
lgbm_model.fit(X_train, y_train)
xgb_model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=500, n_jobs=-1,
              num_parallel_tree=None, random_state=42, ...)

In [122]:
# Evaluation
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    print(y_proba)
    auc_score = roc_auc_score(y_test, y_proba)
    # report = classification_report(y_test, y_pred)
    # cm = confusion_matrix(y_test, y_pred)

    metrics = {
        "roc_auc": roc_auc_score(y_test, y_proba),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1_score": f1_score(y_test, y_pred, zero_division=0)
    }

    evaluation =   {
        "AUC Score": auc_score
    }

    return evaluation, metrics


In [123]:
evaluate_data = []

evaluate_data.append({
    "Model": "LightGBM",
    **evaluate_model(lgbm_model, X_test, y_test)[0]
})

evaluate_data.append({
    "Model": "XGBoost",
    **evaluate_model(xgb_model, X_test, y_test)[0]
})

pd.DataFrame(evaluate_data).sort_values(by="AUC Score", ascending=False)

[0.22078162 0.03334653 0.13970042 ... 0.18091476 0.06563808 0.87265334]
[0.16932356 0.0135582  0.04687255 ... 0.06782644 0.02022734 0.7420998 ]


,Model,AUC Score
1,XGBoost,0.745289
0,LightGBM,0.734264


In [102]:
xgb_model.save_model("model/xgboost_model.json")

In [121]:
evaluation, metrics = evaluate_model(xgb_model, X_test, y_test)
with mlflow.start_run(run_name="xgboost_baseline"):

    mlflow.log_params(xgb_params)
    mlflow.log_metrics(metrics)

    mlflow.set_tag("model_type", "XGBoost")
    mlflow.set_tag("dataset", "customer_propensity")

    # Save model to MLflow
    model_info = mlflow.xgboost.log_model(
        xgb_model=xgb_model,
        name="xgboost_model"
    )

    print("Metrics:", metrics)
    print("Model URI:", model_info.model_uri)

c:\Users\lixx\AppData\Local\Programs\Python\Python39\lib\site-packages\xgboost\core.py:158: UserWarning: [17:36:17] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\c_api\c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)
2026/10/24 17:36:30 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Metrics: {'roc_auc': np.float64(0.7452889696543592), 'precision': 0.5597014925373134, 'recall': 0.2568493150684932, 'f1_score': 0.352112676056338}
Model URI: models:/m-d3d8cdfa5e12424fb9b5a6d6393ba125


# Feature Importance

In [103]:
importance_xgb = pd.Series(
    xgb_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(importance_xgb.head(20))

NumOfProducts      0.599335
IsActiveMember     0.123105
Balance            0.083900
CreditScore        0.070099
EstimatedSalary    0.066824
Tenure             0.056737
dtype: float32


In [105]:
# MLFLOW
# Create an experiment
mlflow.set_experiment("MLFlow Customer Churn Experiment")

2026/10/24 17:28:46 INFO mlflow.tracking.fluent: Experiment with name 'MLFlow Customer Churn Experiment' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///c:/Users/lixx/OneDrive/Documents/Projects/customer_bank_churn/mlruns/205440621418848284', creation_time=1792837726827, experiment_id='205440621418848284', last_update_time=1792837726827, lifecycle_stage='active', name='MLFlow Customer Churn Experiment', tags={}>

# Predicting New Data